# 04 · SQL data model: build, load and reconcile in MySQL 8

**Project:** BYD France – Business Intelligence & Market Analytics · **Phase 4, step 1**

### Purpose
Move the validated Phase 2 datasets into a **relational star schema in MySQL**, and prove the database reproduces the Python results exactly. Power BI will later read from this model.

| Script (in `sql/`) | What it does |
|---|---|
| `01_schema.sql` | Creates the star schema: `dim_brand`, `dim_date`, `fact_market_month`, `fact_brand_month`, with keys and CHECK constraints that enforce the Phase 2 rules |
| `02_load.sql` | Loads the three v2 CSVs unchanged; empty values become **NULL, never 0** |
| `03_checks.sql` | 22 reconciliation checks against the validated files and the Notebook 03 figures; every row must say `PASS` |

### Why a star schema?
* **Facts** are the measured events (registrations per brand per month, market total per month). **Dimensions** describe them (brand ownership, calendar).
* The market total is stored **once per month** (`fact_market_month`), not repeated on 1 734 brand rows. Shares are calculated by joining on `date_key`.
* This is the shape Power BI expects: one-to-many relationships from each dimension to the facts.

### How to run
1. **Runtime → Run all.** Step 1 installs a real MySQL server inside this Colab session (about 1 minute; nothing is installed on your Mac).
2. When step 2 asks, select `brand_month.csv`, `market_month.csv` and `dim_brand.csv` from `data/processed/`.


## 1 · Install and start MySQL inside Colab
**What you should see:** a MySQL version starting with `8.0`. The last command enables `LOCAL INFILE`, which `02_load.sql` needs to read the CSV files.

In [ ]:
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!service mysql start
!mysql -e "SET GLOBAL local_infile = 1; SELECT VERSION() AS mysql_version, @@local_infile AS local_infile_enabled;"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld        su: warning: cannot change directory to /nonexistent: No such file or directory
[ OK ]
+-------------------------+----------------------+
| mysql_version           | local_infile_enabled |
+-------------------------+----------------------+
| 8.0.46-0ubuntu0.24.04.5 |                    1 |
+-------------------------+----------------------+


## 2 · Upload the validated CSVs and verify their fingerprints
Same SHA-256 check as Notebook 03: proves these are the unmodified Phase 2 v2 files.

In [ ]:
import hashlib, io, subprocess
import pandas as pd

EXPECTED_SHA256 = {
    "brand_month.csv":  "75c9a78bbe244ce9577894190f5f41fbb38daaf69011726edb2a895e745013ad",
    "market_month.csv": "7f0b6a204092b78e96f1cda41d8766e07545404c09196f4002dc9bea1367b017",
    "dim_brand.csv":    "d34bfc04a27d3038852513c1b00e26dcde4f1e479458ad87e9d386588de4d442",
}
try:
    from google.colab import files
    files.upload()
except ImportError:
    pass
for name, sha in EXPECTED_SHA256.items():
    ok = hashlib.sha256(open(name, "rb").read()).hexdigest() == sha
    print(f"{name:<18} matches validated v2: {ok}")
    assert ok, f"{name} is not the validated v2 file"

def mysql(sql_file=None, query=None, db="byd_france", local_infile=False):
    """Run a .sql file or a query with the mysql client; return the result as a DataFrame list."""
    cmd = ["mysql", "-B"] + (["--local-infile=1"] if local_infile else []) + ([db] if db else [])
    cmd += ["-e", query] if query else []
    res = subprocess.run(cmd, stdin=open(sql_file) if sql_file else None, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(res.stderr)
    blocks, cur = [], []                      # split the output into one table per SELECT
    for line in res.stdout.splitlines():
        if cur and line.split("\t")[0] in ("loaded_table", "id"):
            blocks.append(cur); cur = []
        cur.append(line)
    if cur: blocks.append(cur)
    return [pd.read_csv(io.StringIO("\n".join(b)), sep="\t") for b in blocks]

brand_month.csv    matches validated v2: True
market_month.csv   matches validated v2: True
dim_brand.csv      matches validated v2: True


## 3 · Write the SQL scripts
These cells write the scripts into the Colab session. They are **identical to the files in `sql/` on GitHub**, so the repository and this notebook can't drift apart.

In [ ]:
%%writefile 01_schema.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 01_schema.sql            (MySQL 8.0.16+)
-- Star schema for the validated Phase 2 v2 datasets.
--
--   dim_brand  (51)     dim_date (34)
--        \               /
--      fact_brand_month (1 734 = 34 months x 51 brands)
--                 |
--      fact_market_month (34)   <- market totals, one row per month
--
-- Design rules
--   * Missing stays missing: registrations are NULL when CCFA did not list
--     the brand that month (value_type = 'not_listed'). Never 0.
--   * The market total lives only in fact_market_month (not repeated on
--     every brand row); join on date_key when a share is needed.
--   * CHECK constraints encode the Phase 2 rules, so bad rows are rejected.
-- Re-running this script drops and rebuilds everything.
-- =====================================================================

CREATE DATABASE IF NOT EXISTS byd_france
  CHARACTER SET utf8mb4 COLLATE utf8mb4_0900_ai_ci;
USE byd_france;

DROP TABLE IF EXISTS fact_brand_month;
DROP TABLE IF EXISTS fact_market_month;
DROP TABLE IF EXISTS dim_date;
DROP TABLE IF EXISTS dim_brand;

-- ---------------------------------------------------------------------
-- dim_brand: one row per brand label used in the CCFA tables
-- ---------------------------------------------------------------------
CREATE TABLE dim_brand (
  brand                      VARCHAR(40)  NOT NULL,
  brand_origin_country       VARCHAR(40)  NULL,      -- NULL only for AUTRES (residual group)
  controlling_owner          VARCHAR(80)  NULL,
  controlling_owner_country  VARCHAR(40)  NULL,
  ownership_structure        VARCHAR(40)  NULL,
  is_chinese_owned           BOOLEAN      NOT NULL,  -- controlling owner (>50% / sole control) HQ in China
  is_chinese_origin          BOOLEAN      NOT NULL,  -- brand heritage, separate from ownership
  is_joint_venture           BOOLEAN      NOT NULL,
  is_byd                     BOOLEAN      NOT NULL,
  classification_note        VARCHAR(255) NULL,
  classification_rule        VARCHAR(255) NOT NULL,
  PRIMARY KEY (brand)
);

-- ---------------------------------------------------------------------
-- dim_date: one row per reporting month (CCFA publishes monthly)
-- ---------------------------------------------------------------------
CREATE TABLE dim_date (
  date_key          DATE        NOT NULL,   -- first day of the month
  period            CHAR(7)     NOT NULL,   -- 'YYYY-MM'
  year              SMALLINT    NOT NULL,
  month             TINYINT     NOT NULL,
  month_name        CHAR(3)     NOT NULL,
  quarter           TINYINT     NOT NULL,
  months_in_data    TINYINT     NOT NULL,   -- months of that year present in the dataset
  is_complete_year  BOOLEAN     NOT NULL,   -- TRUE for 2024 and 2025; FALSE for 2026 (Jan-Oct)
  PRIMARY KEY (date_key),
  UNIQUE KEY uq_period (period),
  CONSTRAINT chk_month CHECK (month BETWEEN 1 AND 12),
  CONSTRAINT chk_first_day CHECK (DAY(date_key) = 1)
);

-- Generate Jan 2024 .. Oct 2026 with a recursive CTE (no hard-coded rows).
INSERT INTO dim_date (date_key, period, year, month, month_name, quarter, months_in_data, is_complete_year)
WITH RECURSIVE m AS (
  SELECT DATE('2024-01-01') AS d
  UNION ALL
  SELECT d + INTERVAL 1 MONTH FROM m WHERE d < '2026-10-01'
)
SELECT d,
       DATE_FORMAT(d, '%Y-%m'),
       YEAR(d),
       MONTH(d),
       DATE_FORMAT(d, '%b'),
       QUARTER(d),
       COUNT(*) OVER (PARTITION BY YEAR(d)),
       COUNT(*) OVER (PARTITION BY YEAR(d)) = 12
FROM m;

-- ---------------------------------------------------------------------
-- fact_market_month: CCFA total French new passenger-car registrations
-- ---------------------------------------------------------------------
CREATE TABLE fact_market_month (
  date_key                    DATE         NOT NULL,
  market_registrations_month  INT          NOT NULL,
  market_registrations_ytd    INT          NOT NULL,   -- CCFA published YTD (never re-summed)
  ytd_revision_gap            INT          NULL,       -- published YTD - (previous YTD + month)
  value_type                  VARCHAR(12)  NOT NULL,
  source_id                   VARCHAR(10)  NOT NULL,   -- -> docs/source_register.csv
  source_file                 VARCHAR(80)  NOT NULL,
  PRIMARY KEY (date_key),
  CONSTRAINT fk_mkt_date FOREIGN KEY (date_key) REFERENCES dim_date (date_key),
  CONSTRAINT chk_mkt_positive CHECK (market_registrations_month > 0 AND market_registrations_ytd > 0)
);

-- ---------------------------------------------------------------------
-- fact_brand_month: one row per brand per month (grain = date_key x brand)
-- ---------------------------------------------------------------------
CREATE TABLE fact_brand_month (
  date_key             DATE          NOT NULL,
  brand                VARCHAR(40)   NOT NULL,
  brand_label          VARCHAR(40)   NULL,       -- label exactly as printed by CCFA
  registrations_month  INT           NULL,       -- NULL = not listed (unknown), never 0
  registrations_ytd    INT           NULL,       -- CCFA published YTD
  ytd_revision_gap     INT           NULL,
  share_month_pct      DECIMAL(6,3)  NULL,       -- CCFA published share, %
  share_ytd_pct        DECIMAL(6,3)  NULL,
  value_type           VARCHAR(12)   NOT NULL,   -- least reliable of the two below
  value_type_month     VARCHAR(12)   NOT NULL,
  value_type_ytd       VARCHAR(12)   NOT NULL,
  listed_in_source     BOOLEAN       NOT NULL,
  in_market_sum        BOOLEAN       NOT NULL,   -- TRUE = part of the comparison universe
  source_id            VARCHAR(10)   NOT NULL,
  source_file          VARCHAR(80)   NOT NULL,
  note                 VARCHAR(255)  NULL,
  PRIMARY KEY (date_key, brand),
  KEY ix_brand (brand),
  KEY ix_universe (in_market_sum, date_key),
  CONSTRAINT fk_fbm_date  FOREIGN KEY (date_key) REFERENCES dim_date (date_key),
  CONSTRAINT fk_fbm_brand FOREIGN KEY (brand)    REFERENCES dim_brand (brand),
  CONSTRAINT chk_value_type CHECK (
        value_type       IN ('actual','aggregated','imputed','not_listed')
    AND value_type_month IN ('actual','aggregated','imputed','not_listed')
    AND value_type_ytd   IN ('actual','aggregated','imputed','not_listed')),
  -- missing stays missing: not_listed <=> no registration figures
  CONSTRAINT chk_not_listed_is_null CHECK (
        (value_type =  'not_listed' AND registrations_month IS NULL AND registrations_ytd IS NULL AND share_month_pct IS NULL)
     OR (value_type <> 'not_listed' AND registrations_month IS NOT NULL)),
  CONSTRAINT chk_universe_listed CHECK (in_market_sum = FALSE OR value_type <> 'not_listed'),
  CONSTRAINT chk_non_negative CHECK (registrations_month >= 0 AND registrations_ytd >= 0)
);


Writing 01_schema.sql


In [ ]:
%%writefile 02_load.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 02_load.sql              (MySQL 8.0)
-- Loads the three validated Phase 2 v2 CSVs (unchanged) into the schema.
--
-- Run from the folder that holds the CSVs, with LOCAL INFILE enabled:
--   mysql --local-infile=1 byd_france < 02_load.sql
--
-- Conversion rules
--   * Empty CSV field  -> NULL  (NULLIF(@x, '')). Never 0.
--   * '105.0' (pandas float export) -> 105 via DECIMAL, exact because every
--     count is a whole number (checked in Phase 2).
--   * 'True' / 'False' -> 1 / 0.
-- IMPORTANT: LOAD DATA LOCAL turns data errors into WARNINGS and skips the
-- bad row instead of failing. So each load is followed by a warning count,
-- and 03_checks.sql verifies every row count. Expected warnings: 0.
-- =====================================================================

USE byd_france;
SET FOREIGN_KEY_CHECKS = 1;

-- Remove previously loaded facts so the script can be re-run safely (dim_date is kept).
DELETE FROM fact_brand_month;
DELETE FROM fact_market_month;
DELETE FROM dim_brand;

-- ---------------------------------------------------------------- dim_brand
LOAD DATA LOCAL INFILE 'dim_brand.csv'
INTO TABLE dim_brand
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(brand, @origin, @owner, @owner_country, @structure,
 @cn_owned, @cn_origin, @jv, @is_byd, @note, classification_rule)
SET brand_origin_country      = NULLIF(@origin, ''),
    controlling_owner         = NULLIF(@owner, ''),
    controlling_owner_country = NULLIF(@owner_country, ''),
    ownership_structure       = NULLIF(@structure, ''),
    is_chinese_owned          = (@cn_owned  = 'True'),
    is_chinese_origin         = (@cn_origin = 'True'),
    is_joint_venture          = (@jv        = 'True'),
    is_byd                    = (@is_byd    = 'True'),
    classification_note       = NULLIF(@note, '');
SELECT 'dim_brand' AS loaded_table, ROW_COUNT() AS rows_loaded, @@warning_count AS warnings;

-- -------------------------------------------------------- fact_market_month
LOAD DATA LOCAL INFILE 'market_month.csv'
INTO TABLE fact_market_month
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(@period, @date, @year, @month, @mkt_m, @mkt_ytd, @gap, value_type, source_id, source_file)
SET date_key                   = @date,
    market_registrations_month = CAST(@mkt_m   AS DECIMAL(12,1)),
    market_registrations_ytd   = CAST(@mkt_ytd AS DECIMAL(12,1)),
    ytd_revision_gap           = CAST(NULLIF(@gap, '') AS DECIMAL(12,1));
SELECT 'fact_market_month' AS loaded_table, ROW_COUNT() AS rows_loaded, @@warning_count AS warnings;

-- --------------------------------------------------------- fact_brand_month
-- The CSV also carries period/year/month and the market totals; they are
-- read into variables and not stored (dim_date and fact_market_month hold them).
LOAD DATA LOCAL INFILE 'brand_month.csv'
INTO TABLE fact_brand_month
CHARACTER SET utf8mb4
FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES
(@period, @date, @year, @month, brand, @label, @reg_m, @reg_ytd, @gap,
 @mkt_m, @mkt_ytd, @sh_m, @sh_ytd, value_type, value_type_month, value_type_ytd,
 @listed, @in_sum, source_id, source_file, @note)
SET date_key            = @date,
    brand_label         = NULLIF(@label, ''),
    registrations_month = CAST(NULLIF(@reg_m,   '') AS DECIMAL(12,1)),
    registrations_ytd   = CAST(NULLIF(@reg_ytd, '') AS DECIMAL(12,1)),
    ytd_revision_gap    = CAST(NULLIF(@gap,     '') AS DECIMAL(12,1)),
    share_month_pct     = NULLIF(@sh_m,   ''),
    share_ytd_pct       = NULLIF(@sh_ytd, ''),
    listed_in_source    = (@listed = 'True'),
    in_market_sum       = (@in_sum = 'True'),
    note                = NULLIF(@note, '');
SELECT 'fact_brand_month' AS loaded_table, ROW_COUNT() AS rows_loaded, @@warning_count AS warnings;


Writing 02_load.sql


In [ ]:
%%writefile 03_checks.sql
-- =====================================================================
-- BYD France BI · Phase 4 · 03_checks.sql            (MySQL 8.0)
-- Reconciliation checks after loading. Every row must return 'PASS'.
-- The expected values come from the validated Phase 2 files and the
-- Notebook 03 results, so passing proves SQL and Python agree.
--   mysql -t byd_france < 03_checks.sql
-- =====================================================================

USE byd_france;

WITH
universe AS (                               -- comparison universe: no Fiat/Abarth components, no not_listed rows
  SELECT * FROM fact_brand_month WHERE in_market_sum = 1
),
u_month AS (
  SELECT u.date_key,
         SUM(u.registrations_month) AS u_month,
         SUM(u.registrations_ytd)   AS u_ytd
  FROM universe u
  GROUP BY u.date_key
),
checks AS (
  SELECT 'K01' AS id, 'dim_brand rows' AS check_name, 51 AS expected,
         (SELECT COUNT(*) FROM dim_brand) AS actual
  UNION ALL SELECT 'K02', 'dim_date rows (Jan 2024 - Oct 2026)', 34, (SELECT COUNT(*) FROM dim_date)
  UNION ALL SELECT 'K03', 'fact_market_month rows', 34, (SELECT COUNT(*) FROM fact_market_month)
  UNION ALL SELECT 'K04', 'fact_brand_month rows (34 x 51)', 1734, (SELECT COUNT(*) FROM fact_brand_month)
  UNION ALL SELECT 'K05', 'months without exactly 51 brand rows', 0,
         (SELECT COUNT(*) FROM (SELECT date_key FROM fact_brand_month GROUP BY date_key HAVING COUNT(*) <> 51) x)
  UNION ALL SELECT 'K06', 'months where universe <> market total (month)', 0,
         (SELECT COUNT(*) FROM fact_market_month m LEFT JOIN u_month u ON u.date_key = m.date_key
          WHERE u.u_month IS NULL OR u.u_month <> m.market_registrations_month)
  UNION ALL SELECT 'K07', 'months where universe <> market total (YTD)', 0,
         (SELECT COUNT(*) FROM fact_market_month m LEFT JOIN u_month u ON u.date_key = m.date_key
          WHERE u.u_ytd IS NULL OR u.u_ytd <> m.market_registrations_ytd)
  UNION ALL SELECT 'K08', 'Fiat / Abarth component rows in universe', 0,
         (SELECT COUNT(*) FROM universe WHERE brand IN ('Fiat', 'Abarth'))
  UNION ALL SELECT 'K09', 'not_listed rows with any figure (zero-fill)', 0,
         (SELECT COUNT(*) FROM fact_brand_month WHERE value_type = 'not_listed'
          AND (registrations_month IS NOT NULL OR registrations_ytd IS NOT NULL OR share_month_pct IS NOT NULL))
  UNION ALL SELECT 'K10', 'not_listed rows in universe', 0,
         (SELECT COUNT(*) FROM universe WHERE value_type = 'not_listed')
  UNION ALL SELECT 'K11', 'NULL monthly registrations preserved (as in CSV)', 260,
         (SELECT COUNT(*) FROM fact_brand_month WHERE registrations_month IS NULL)
  UNION ALL SELECT 'K12', 'zero registrations (CCFA-published zeros only)', 6,
         (SELECT COUNT(*) FROM fact_brand_month WHERE registrations_month = 0)
  UNION ALL SELECT 'K13', 'universe rows without an ownership classification', 0,
         (SELECT COUNT(*) FROM universe u LEFT JOIN dim_brand b ON b.brand = u.brand WHERE b.brand IS NULL)
  UNION ALL SELECT 'K14', 'Chinese-owned brands in dim_brand', 12,
         (SELECT COUNT(*) FROM dim_brand WHERE is_chinese_owned = 1)
  UNION ALL SELECT 'K15', 'complete-year months in dim_date (2024 + 2025)', 24,
         (SELECT COUNT(*) FROM dim_date WHERE is_complete_year = 1)
  UNION ALL SELECT 'K16', 'BYD full year 2024 (Dec published YTD)', 5415,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2024-12-01')
  UNION ALL SELECT 'K17', 'BYD full year 2025 (Dec published YTD)', 13533,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2025-12-01')
  UNION ALL SELECT 'K18', 'BYD Jan-Oct 2025 (published YTD)', 9843,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2025-10-01')
  UNION ALL SELECT 'K19', 'BYD Jan-Oct 2026 (published YTD)', 28086,
         (SELECT registrations_ytd FROM fact_brand_month WHERE brand = 'BYD' AND date_key = '2026-10-01')
  UNION ALL SELECT 'K20', 'Market full year 2025 (Dec published YTD)', 1632154,
         (SELECT market_registrations_ytd FROM fact_market_month WHERE date_key = '2025-12-01')
  UNION ALL SELECT 'K21', 'Market Jan-Oct 2026 (published YTD)', 1384293,
         (SELECT market_registrations_ytd FROM fact_market_month WHERE date_key = '2026-10-01')
  UNION ALL SELECT 'K22', 'Xpeng months with figures (listed from Mar 2025)', 20,
         (SELECT COUNT(registrations_month) FROM fact_brand_month WHERE brand = 'Xpeng')
)
SELECT id, check_name, expected, actual,
       CASE WHEN actual = expected THEN 'PASS' ELSE 'FAIL' END AS result
FROM checks
ORDER BY id;


Writing 03_checks.sql


## 4 · Build the schema
**What it does:** drops and recreates the database objects, and generates `dim_date` with a recursive CTE.
**What you should see:** 34 calendar months, with `is_complete_year` = 1 for 2024–2025 and 0 for 2026.

In [ ]:
mysql("01_schema.sql", db=None)
mysql(query="SELECT year, COUNT(*) AS months, MIN(period) AS first_month, MAX(period) AS last_month, "
            "MAX(is_complete_year) AS is_complete_year FROM dim_date GROUP BY year ORDER BY year")[0]

,year,months,first_month,last_month,is_complete_year
0,2024,12,2024-01,2024-12,1
1,2025,12,2025-01,2025-12,1
2,2026,10,2026-01,2026-10,0


## 5 · Load the data
**Watch the `warnings` column.** `LOAD DATA LOCAL` doesn't stop on bad data; it skips the row and raises a warning. So **0 warnings** and the right row counts are both required.

In [ ]:
loads = pd.concat(mysql("02_load.sql", local_infile=True), ignore_index=True)
display(loads)
expected_rows = {"dim_brand": 51, "fact_market_month": 34, "fact_brand_month": 1734}
assert (loads.warnings == 0).all(), "load produced warnings: some rows were changed or skipped"
assert dict(zip(loads.loaded_table, loads.rows_loaded)) == expected_rows, "unexpected row counts"
print("Load OK: expected row counts, 0 warnings.")

,loaded_table,rows_loaded,warnings
0,dim_brand,51,0
1,fact_market_month,34,0
2,fact_brand_month,1734,0


Load OK: expected row counts, 0 warnings.


## 6 · Reconciliation checks
22 checks covering row counts, the universe-equals-market rule (monthly and YTD), no double counting, **NULL preserved (260 rows) and no invented zeros**, and headline figures from Notebook 03 (BYD 5 415 / 13 533 / 9 843 / 28 086; market 1 632 154 / 1 384 293).

In [ ]:
checks = mysql("03_checks.sql")[0]
display(checks)
print("ALL CHECKS PASS:", bool((checks.result == "PASS").all()))
assert (checks.result == "PASS").all()

,id,check_name,expected,actual,result
0,K01,dim_brand rows,51,51,PASS
1,K02,dim_date rows (Jan 2024 - Oct 2026),34,34,PASS
2,K03,fact_market_month rows,34,34,PASS
3,K04,fact_brand_month rows (34 x 51),1734,1734,PASS
4,K05,months without exactly 51 brand rows,0,0,PASS
5,K06,months where universe <> market total (month),0,0,PASS
6,K07,months where universe <> market total (YTD),0,0,PASS
7,K08,Fiat / Abarth component rows in universe,0,0,PASS
8,K09,not_listed rows with any figure (zero-fill),0,0,PASS
9,K10,not_listed rows in universe,0,0,PASS


ALL CHECKS PASS: True


## 7 · Prove the constraints are enforced by MySQL
Three deliberately invalid writes. **Each must be rejected** by the database itself, not by Python:
1. a `not_listed` row given a value of 0 (zero-fill),
2. a brand that doesn't exist in `dim_brand` (foreign key),
3. a `not_listed` row added to the comparison universe.

In [ ]:
bad_writes = {
    "zero-fill on a not_listed row":
        "INSERT INTO fact_brand_month (date_key, brand, registrations_month, value_type, value_type_month, value_type_ytd, "
        "listed_in_source, in_market_sum, source_id, source_file) "
        "VALUES ('2024-01-01', 'Ferrari', 0, 'not_listed', 'not_listed', 'not_listed', 0, 0, 'TEST', 'test')",
    "unknown brand (foreign key)":
        "INSERT INTO fact_brand_month (date_key, brand, registrations_month, value_type, value_type_month, value_type_ytd, "
        "listed_in_source, in_market_sum, source_id, source_file) "
        "VALUES ('2024-01-01', 'NoSuchBrand', 1, 'actual', 'actual', 'actual', 1, 1, 'TEST', 'test')",
    "not_listed row in the universe":
        "UPDATE fact_brand_month SET in_market_sum = 1 WHERE value_type = 'not_listed' LIMIT 1",
}
for label, q in bad_writes.items():
    try:
        mysql(query=q)
        print(f"NOT REJECTED: {label}")
        raise AssertionError(label)
    except RuntimeError as e:
        print(f"rejected as expected: {label:<32} -> {str(e).strip().splitlines()[-1][:90]}")
# the data is untouched after the rejected writes
assert (mysql("03_checks.sql")[0].result == "PASS").all()
print("Data unchanged: all 22 checks still PASS.")

rejected as expected: zero-fill on a not_listed row    -> ERROR 3819 (HY000) at line 1: Check constraint 'chk_not_listed_is_null' is violated.
rejected as expected: unknown brand (foreign key)      -> ERROR 1452 (23000) at line 1: Cannot add or update a child row: a foreign key constraint f
rejected as expected: not_listed row in the universe   -> ERROR 3819 (HY000) at line 1: Check constraint 'chk_universe_listed' is violated.
Data unchanged: all 22 checks still PASS.


## 8 · The model at a glance

In [ ]:
mysql(query="""
SELECT tc.table_name, tc.constraint_type, COUNT(*) AS n_constraints
FROM information_schema.table_constraints tc
WHERE tc.table_schema = 'byd_france'
GROUP BY tc.table_name, tc.constraint_type
ORDER BY tc.table_name, tc.constraint_type""")[0]

,TABLE_NAME,CONSTRAINT_TYPE,n_constraints
0,dim_brand,PRIMARY KEY,1
1,dim_date,CHECK,2
2,dim_date,PRIMARY KEY,1
3,dim_date,UNIQUE,1
4,fact_brand_month,CHECK,4
5,fact_brand_month,FOREIGN KEY,2
6,fact_brand_month,PRIMARY KEY,1
7,fact_market_month,CHECK,1
8,fact_market_month,FOREIGN KEY,1
9,fact_market_month,PRIMARY KEY,1


## Summary
* The four-table star schema is built in **MySQL 8**, with primary keys, foreign keys and CHECK constraints that enforce the Phase 2 rules in the database itself.
* The three validated CSVs load **unchanged, with 0 warnings**; the 260 unlisted values are NULL and no zero was created.
* All **22 reconciliation checks pass**, so the database reproduces the Notebook 03 figures exactly.

**Next (Phase 4, step 2):** `04_views.sql` (reusable views for Power BI) and `05_business_queries.sql` (business questions answered in SQL with CTEs and window functions).